# Introduction

In this exercise we will solve the Cart-Pole problem using on-policy SARSA, off-policy Q learning, and off-policy SARSA with an $\epsilon$-greedy policy.


In [ ]:
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "src" / "rl_suite").is_dir():
        sys.path.insert(0, str(_p / "src"))
        break

import gymnasium as gym

import rl_suite as rl
from rl_suite.utils import RLEnvironmentRunner, print_discrete_space

agent = rl.algorithms()


In [ ]:
ENV_ID, SEED = "CartPole-v1", 10
NUM_TIMESTEPS_GOAL = 10000
env = gym.make(ENV_ID, max_episode_steps=NUM_TIMESTEPS_GOAL)

CARTPOLE_TD_DISCRETIZATION = {
    "num_bins": 14,
    "intervals": ((-2.4, 2.4), (-2.5, 2.5), (-0.2095, 0.2095), (-3.5, 3.5)),
    "include_terminal_bin": True,
}


In [ ]:
# Discretization is configured on ``rl_suite.utils.RLEnvironmentRunner``.


The algorithms we use are temporal-difference methods implemented in `rl_suite.algorithms.td`. TD methods are tabular, so this notebook configures `RLEnvironmentRunner` to discretize the CartPole state space while keeping the algorithm classes independent from CartPole-specific bins. Our MC control algorithms failed to converge earlier, so this time we have decided to substantially increase the size of the state space by increasing the number of bins from $10^4 = 10000$ to $15^4 = 50625$. The loops do take substantially longer to finish as a result.

Aside from this, the basic concepts of the discretization essentially remain the same but with one important difference: The terminal states are now included in the episodes, and therefore must be included in our discrete space. In Assignment 2, the bins were indexed by 0 (first bin was bin 0, second bin was bin 1,...). We have changed this; the first bin is bin 1, second bin is bin 2, and so on. Any terminal value is represented by bin 0 (for each of the state variables). So $n=14$ bins refer to the $n=14$ partitions of the non-terminal values, however in effect the size of our state space is $(n+1)^4 = 15^4$ when accounting for the terminal states.

The TD agents all consume the same discretized runner. Differences between SARSA, Q-learning, and Expected SARSA are now represented by different algorithm classes rather than by notebook-local subclasses.

In [ ]:
td_runner = RLEnvironmentRunner(env, discretization=CARTPOLE_TD_DISCRETIZATION)
print_discrete_space(td_runner.discrete_space)


# Part A: On-Policy SARSA

`SarsaAgent` is imported from `rl_suite.algorithms.td.td_algos.sarsa` and uses the discretized runner configured above.

In [ ]:
# On-policy SARSA is implemented by the package TD module.


In [ ]:
# On-policy SARSA is available as ``agent.td.sarsa()``.


In [ ]:
sarsa_agent = agent.td.sarsa()
sarsa_output = sarsa_agent.control(
    td_runner,
    num_timesteps_goal=NUM_TIMESTEPS_GOAL,
    close_env=False,
)


There results are far more promising than the ones from Assignment 2. We are consistently seeing episodes longer than 100 timesteps. This is partially due to the efficacy of TD algorithms (and learning online concurrently with simulation), and partially due to the increased state space. 

# Part B: Off-Policy Q-learning

`QLearningAgent` is imported from `rl_suite.algorithms.td.td_algos.q_learning`; only the TD update rule differs from SARSA.

In [ ]:
# Q-learning is implemented by the package TD module.


In [ ]:
q_learning_agent = agent.td.q_learning()
q_learning_output = q_learning_agent.control(
    td_runner,
    num_timesteps_goal=NUM_TIMESTEPS_GOAL,
    close_env=False,
)


# Part C: Off-Policy Expected SARSA

`ExpectedSarsaAgent` is imported from `rl_suite.algorithms.td.td_algos.expected_sarsa`. Here, the target policy is chosen to be an epsilon-greedy policy even though this is an off-policy method. The reason this is done is so that the update rule can take the expected value of the next possible state-action pairs. Since our target policy is $\epsilon$-greedy with $\epsilon=0.15$, we chose a similarly $\epsilon$-greedy behaviour policy which is more exploratory ($\epsilon_b = \epsilon*2 = 0.3$). The requirement of coverage is still clearly met.

In [ ]:
# Expected SARSA is implemented by the package TD module.


In [ ]:
expected_sarsa_agent = agent.td.expected_sarsa()
expected_sarsa_output = expected_sarsa_agent.control(
    td_runner,
    num_timesteps_goal=NUM_TIMESTEPS_GOAL,
    close_env=False,
)


# Discussion

TD Algorithms are far more suited to this problem than naive MC-based control techniques. These algorithms show promising results for convergence (though they are still quite inefficient and will take a long time to reach that point, if ever). SARSA and Q-learning seem to perform better than Expected SARSA. However, Expected SARSA took less than half as much the time to finish the loop as the others. This is consistent with our theoretical knowledge: Expected SARSA uses an epsilon-greedy policy, and not a true optimal policy, but this allows us to use expected value in the update step, which provides drastic improvement in computational efficiency).